In [1]:
import os
import pickle
import numpy as np
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image

In [2]:
%pip install tqdm

   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   ---------------------------------------- 676.7/676.7 kB 3.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
DATASET_PATH = r"C:\Users\prave\OneDrive\Desktop\jupyter notebook 333\skin_type_classification_dataset"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")

In [3]:
feature_model = EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    pooling="avg",
    input_shape=(224,224,3)
)

print("Feature extractor loaded.")

Feature extractor loaded.


In [4]:
def extract_feature(img_path):

    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)

    feature = feature_model.predict(img, verbose=0)

    return feature.flatten()

In [5]:
features = []
image_paths = []
labels = []

classes = sorted(os.listdir(TRAIN_DIR))

for cls in classes:

    class_folder = os.path.join(TRAIN_DIR, cls)

    if not os.path.isdir(class_folder):
        continue

    print(f"Processing {cls}...")

    for file in tqdm(os.listdir(class_folder)):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            img_path = os.path.join(class_folder, file)

            feature = extract_feature(img_path)

            features.append(feature)
            image_paths.append(img_path)
            labels.append(cls)

Processing combination...


  0%|          | 0/248 [00:00<?, ?it/s]

100%|██████████| 248/248 [05:59<00:00,  1.45s/it]


Processing dry...


100%|██████████| 833/833 [20:54<00:00,  1.51s/it]  


Processing normal...


100%|██████████| 815/815 [17:20<00:00,  1.28s/it]


In [6]:
features = np.array(features)

print("Feature Shape:", features.shape)

Feature Shape: (2872, 1280)


In [7]:
os.makedirs("features", exist_ok=True)

np.save("features/feature_vectors.npy", features)

with open("features/image_paths.pkl","wb") as f:
    pickle.dump(image_paths,f)

with open("features/labels.pkl","wb") as f:
    pickle.dump(labels,f)

print("Dataset features saved successfully!")

Dataset features saved successfully!


In [8]:
print("Files created:")

print("features/feature_vectors.npy")
print("features/image_paths.pkl")
print("features/labels.pkl")

Files created:
features/feature_vectors.npy
features/image_paths.pkl
features/labels.pkl
